# Analysis of Benchmark Results for 100 Single-Shot _v9_combined_23_disrupt_ HDF5 Files and Equivalent Zarr Stores with Alcator C-Mod Data

This notebook benchmarks reading the Alcator C-Mod shot data for the _v9_combined_23_disrupt_ MIT signal selection from two storage formats -- HDF5 files and Zarr v3 stores converted from those HDF5 files -- on two AWS EC2 instances and two storage locations. This top cell is a self-contained TL;DR that summarises the entire notebook; the analysis sections that follow it provide the per-format details, the side-by-side comparison plots, and all benchmark numbers.

## Data: HDF5 Files

Each HDF5 file was created with the following properties:

* Each file holds approximately 370 signals from a single shot. The signal selection was done by the MIT team and designated as _v9_combined_23_disrupt_.
* Cloud optimized using the paged aggregation file space management with the file page size of 8,000,000 bytes (8 MB).
* Size of each file is exactly 64,000,000 bytes (64 MB) due to the selected file page size.
* The files have same HDF5 group hierarchy: `/shots/<SHOT_ID>/signals/<signal MDSplus expression>`.
* The MDSplus `dim_of` data are stored as HDF5 dimension scale datasets in the `/shots/<SHOT_ID>/` group and attached to the appropriate dimensions of the signal HDF5 datasets.
* Only unique dimension scales are stored by comparing the MD5 checksum of dimension scale values. This means there could be multiple signal datasets that share the same dimension scale.
* HDF5 datasets (dimension scales and signals) with total size greater than 8 kB are compressed with the Deflate (a.k.a. _gzip_) compression at level 4.

## Data: Zarr Stores

The same shot data were also stored as Zarr v3 stores. Each shot is a single self-contained Zarr store with consolidated metadata. Sharding is enabled so that, for arrays that were chunked in HDF5, all chunks of an array are bundled into a single shard. The per-array chunk shapes and the GZip compression are preserved from the source HDF5 datasets. Xarray-style dimension coordinates are recorded via the `_ARRAY_DIMENSIONS` attribute. The benchmark code reads the stores with `zarr-python`, using the `obstore`-based `zarr.storage.ObjectStore` wrapper for both the local file system (`LocalStore`) and S3 (`S3Store`).

There is no equivalent of an HDF5 file page cache in this Zarr setup, so the `pb-size` axis from the HDF5 analysis is dropped from the Zarr analysis below. The two access patterns and the two EC2 instances are otherwise identical to the HDF5 case.

## Benchmarks

The benchmarks were run on two AWS EC2 instances:

* `m5.4xlarge` with 16 vCPU (8 CPU cores, 2 threads per core), 64 GiB memory, up to 10 Gbps network bandwidth.
* `m7i.48xlarge` with 192 vCPUs, 768 GiB memory, 50 Gbps network bandwidth, 40 Gbps EBS bandwidth.

The benchmark cases were created by combining the following parameters:

* 100 HDF5 files (or, for the Zarr variant, 100 Zarr stores).
* The files / stores were placed either in the local file system (EBS) or in an
  S3 bucket. HDF5 files in S3 were read via the HDF5 library's Read-Only S3
  (ROS3) driver; Zarr stores were read via `zarr-python` over `obstore`
  (`LocalStore` / `S3Store`).
* Data reading tasks were shared among a range of Dask workers, never to exceed
  one worker per vCPU. The processing tasks were divided using an equal effort
  principle, and the actual number of Dask workers was adjusted accordingly so
  not to have any unused workers.
* HDF5 library page cache size was zero (off, for local files only) or 70 MB.
  Note that since the HDF5 files are all 64 MB in size, the page cache was large
  enough to eventually hold an entire file. Zarr has no equivalent axis.
* Read all signals from all the files / stores in one of two access patterns:
  * One signal at a time from all the files / stores (`obj-type="signals"`). The
    number of files to read a signal from varied per Dask worker depending on
    their total number.
  * Read all signals from one file / store at a time (`obj-type="shots"`). The
    number of files to read all the signals from varied per Dask worker
    depending on their total number.

## TL;DR HDF5 Conclusions

The HDF5 numbers below use the page-cache-off (`pb-size='off'`) slice for local
files -- the no-caching baseline -- and the 70 MB page-cache slice
(`pb-size='70MB'`) for S3 files, which was the only S3 cache size benchmarked.
It is well-known that typical HDF5 files, as well as, accessing cloud optimized
files without page caching yields very poor performance for files in cloud
object stores. Accessing cloud optimized files in a traditional (default) way
when in file systems (i.e., local here) does not impact performance.

For reading all signals from each file as a unit (`obj-type="shots"`):

* `m5.4xlarge`:
  * Local files: total runtime 6.7 to 65.6 s across 1 to 15 Dask workers;
    minimum 6.7 s at 15 workers, ~9.8x speed-up over the 1-worker baseline.
  * S3 files: total runtime 12.6 to 190.9 s across 1 to 15 workers; minimum 12.6
    s at 15 workers, ~15x speed-up.
* `m7i.48xlarge`:
  * Local files: total runtime 1.3 to 41.8 s across 1 to 100 workers; minimum
    1.3 s at 50 workers, ~32x speed-up. Performance keeps improving well past
    the `m5.4xlarge` worker count.
  * S3 files: total runtime 3.0 to 12.0 s across 12 to 100 workers; minimum 3.0
    s at 100 workers, ~3.9x speed-up over the 12-worker baseline. This S3
    benchmark did not run with fewer than 12 workers.

For reading one signal at a time across all 100 files (`obj-type="signals"`):

* `m5.4xlarge`:
  * Local files: total runtime 35.8 to 101.4 s across 1 to 16 workers; minimum
    35.8 s at 6 workers, ~2.8x speed-up over the 1-worker baseline. Performance
    plateaus or worsens past 6-8 workers.
  * S3 files: total runtime 1143 to 16303 s across 1 to 16 workers; minimum 1143
    s at 16 workers, ~14x speed-up. The Aggregated File Open Overhead (AFOO)
    Index stays between 0.55 and 0.63, so opening files dominates total runtime
    in this access pattern.
* `m7i.48xlarge`:
  * Local files: total runtime 23.8 to 276.1 s across 1 to 100 workers; minimum
    23.8 s at 6 workers, ~2.7x speed-up. Performance is essentially flat past
    about 8 workers.
  * S3 files: total runtime 490 to 1284 s across 12 to 100 workers; minimum 490
    s at 90 workers, ~2.6x speed-up over the 12-worker baseline. AFOO ranges
    from 0.18 (at 100 workers) up to 0.59 (at 12 workers), so opening files is
    still a significant fraction of runtime even with many workers.

Per-task read-time estimates from the "One Signal Read Time Estimate" cells (75th-percentile statistic; see those cells for the exact definition):

* `m5.4xlarge` local: ~0.0011 s per signal, ~0.9 s per file.
* `m5.4xlarge` S3: ~0.22 s per signal, ~1.4 s per file.
* `m7i.48xlarge` local: ~0.0007 s per signal, ~0.5 s per file.
* `m7i.48xlarge` S3: ~0.19 s per signal, ~1.1 s per file.

Taking all of the above into consideration:

* The shots-mode access pattern is dramatically more attractive than the
  signals-mode pattern for HDF5: it scales near-linearly with worker count,
  suffers no file-open bottleneck (AFOO well below 0.05 in every shots-mode case
  for HDF5), and reaches single-second runtimes on the `m7i.48xlarge` for all
  100 files in both local and S3 storage.
* The signals-mode pattern is dominated by file-open cost when files are in S3.
  AFOO above 0.5 means the workers spend more than half of their time opening
  files; adding more workers brings limited benefit. Real-world workloads that
  need fan-out per-signal access should aim to minimize the number of opens.
* The larger EC2 instance is well justified for shots-mode workloads, where it
  brings near-linear improvements down to about 3 seconds for all 100 S3 files.
  For signals-mode workloads its added cost is harder to defend, since the gain
  over the `m5.4xlarge` is modest at best.

## TL;DR Zarr Conclusions

For reading all signals from all shot stores, one signal at a time across all
stores (`obj-type="signals"`):

* Local stores:
  * Total runtime spans roughly 121 to 1930 seconds across both EC2 instances.
    The minimum of about 121 seconds was reached at 23 Dask workers on the
    `m7i.48xlarge`.
  * Best speed-up of about 11x on the `m7i.48xlarge` (23 workers) and about 7x
    on the `m5.4xlarge` (16 workers) relative to a single-worker baseline.
  * Beyond ~32 workers on the `m7i.48xlarge` total runtime grows again,
    indicating contention between workers competing for the same stores.
* S3 stores:
  * Total runtime spans roughly 317 to 7993 seconds across the two EC2
    instances. The minimum of about 317 seconds was reached at 32 Dask workers
    on the `m7i.48xlarge`. The `m7i.48xlarge` S3 benchmark did not run with
    fewer than 12 workers.
  * Best speed-up of about 13x on the `m5.4xlarge` (16 workers vs the 1-worker
    baseline) and about 4x on the `m7i.48xlarge` (32 workers vs the 12-worker
    baseline).
  * The Aggregated File Open Overhead (AFOO) Index sits in the 0.45-0.49 range
    on the `m5.4xlarge` and from 0.08 (at 100 workers) up to 0.40 (at 12
    workers) on the `m7i.48xlarge`, so opening Zarr stores is a substantial
    fraction of total runtime in this access pattern.

For reading all signals from all shot stores, all signals from one store at a
time (`obj-type="shots"`):

* Local stores:
  * Total runtime spans roughly 3.85 to 168 seconds across both EC2 instances.
    The minimum of about 3.85 seconds was reached at 100 Dask workers on the
    `m7i.48xlarge`.
  * Speed-up scales near-linearly with worker count, reaching about 34x at 100
    workers on the `m7i.48xlarge`, and about 6x at 15 workers on the
    `m5.4xlarge`.
  * The AFOO Index stays well below 0.05 across all worker counts, so opening
    stores is not a bottleneck in this mode.
* S3 stores:
  * Total runtime spans roughly 34 to 3723 seconds across both EC2 instances.
    The minimum of about 34 seconds was reached at 100 Dask workers on the
    `m7i.48xlarge`. The `m7i.48xlarge` S3 benchmark did not run with fewer than
    12 workers.
  * Speed-up of about 17x on the `m5.4xlarge` (15 workers vs the 1-worker
    baseline) and about 7.5x on the `m7i.48xlarge` (100 workers vs the 12-worker
    baseline).
  * The AFOO Index stays well below 0.01 across all worker counts; per-store
    opens are negligible relative to the data reading time when each store is
    read in full.

For reading all signals from one shot store (per-store read time estimate
derived as in the HDF5 analysis):

* A local store:
  * Estimate of about 3.6 seconds (`m5.4xlarge`) and 2.7 seconds
    (`m7i.48xlarge`) per store.
  * The 75th-percentile median per-store open time is about 0.08 seconds
    (`m5.4xlarge`) and 0.03 seconds (`m7i.48xlarge`); a non-trivial fixed cost
    given how many stores are opened per worker in the per-signal mode.
* An S3 store:
  * Estimate of about 30.8 seconds (`m5.4xlarge`) and 27.9 seconds
    (`m7i.48xlarge`) per store.
  * The 75th-percentile median per-store open time is about 0.12 seconds
    (`m5.4xlarge`) and 0.08 seconds (`m7i.48xlarge`).

Taking all of the above into consideration:

* The two access patterns lead to very different conclusions for the Zarr
  stores. Reading all signals from each store as a unit benefits strongly from
  additional Dask workers and from the larger EC2 instance, with near-linear
  scaling for both local and S3 storage and minimal store-open overhead. Reading
  one signal at a time across all stores is heavily penalised by the cost of
  opening Zarr stores: each worker reopens every store once per signal it is
  responsible for, and this open cost is the dominant component of total runtime
  at low to moderate worker counts.
* Workloads that need to fan out reads of a single signal across many stores
  should aim to minimize store-open cost (by caching opened groups, batching, or
  by aggregating signals across shots into single arrays). Workloads that read
  each store as a unit are the most attractive use of Zarr in this setup.

## TL;DR HDF5 vs Zarr Comparison

The HDF5 vs Zarr comparison section places the two formats side by side per EC2
instance, per storage location, and per access pattern. Best-case end-to-end
runtimes (lowest among all benchmarked worker counts in the relevant
worker-count range) are summarised below; the plots in the comparison section
give the full worker-count dependence.

For reading all signals from each file/store as a unit (`obj-type="shots"`), HDF5 wins on every panel:

* `m5.4xlarge`
  * local: 6.7 s (HDF5) vs 26.5 s (Zarr) -- HDF5 ~4.0x faster.
  * S3:    12.6 s (HDF5) vs 213.7 s (Zarr) -- HDF5 ~17x faster.
* `m7i.48xlarge`
  * local: 1.3 s (HDF5) vs 3.85 s (Zarr) -- HDF5 ~3.0x faster.
  * S3:    3.0 s (HDF5) vs 34.3 s (Zarr) -- HDF5 ~11x faster.

For reading one signal at a time across all files/stores (`obj-type="signals"`),
the picture splits by storage location:

* HDF5 wins on local storage:
  * `m5.4xlarge` local:   35.8 s (HDF5) vs 283 s (Zarr) -- HDF5 ~7.9x faster.
  * `m7i.48xlarge` local: 23.8 s (HDF5) vs 121 s (Zarr) -- HDF5 ~5.1x faster.
* Zarr wins on S3 storage:
  * `m5.4xlarge` S3:    1143 s (HDF5) vs 598 s (Zarr) -- Zarr ~1.9x faster.
  * `m7i.48xlarge` S3:   490 s (HDF5) vs 317 s (Zarr) -- Zarr ~1.5x faster.

Why the split?

* Per-store opens are markedly more expensive for HDF5 when in S3 (0.3-0.5 ms
  locally vs. ~260-280 ms over ROS3 on S3). This extra cost for Zarr is smaller
  (30-80 ms locally vs. 85-115 ms on S3).
* In `obj-type="shots"` mode (one task = all signals from one file/store), each
  worker pays the open cost once per file/store, so the overhead is amortized
  across all the signals it reads. HDF5's single-file cloud optimized format
  then reads through the file very efficiently, and Zarr's per-array sharded
  layout cannot match it.
* In `obj-type="signals"` mode on S3, the HDF5 ROS3 driver re-reads the file's
  internal metadata from S3 for every signal because there is no native cache
  that survives between file opens; the per-signal overhead adds up across 100
  files. Zarr v3 with sharding fetches all chunks of one array in a few large
  GETs once the consolidated metadata is loaded, which is what per-signal-on-S3
  access rewards. This is the only access pattern in the entire benchmark where
  Zarr is the faster format.

Practical takeaways:

* For "load many signals from each shot as a unit" workloads (the common
  analysis-ready pattern), cloud optimized HDF5 is the better format on
  this hardware/setup, regardless of storage location and EC2 instance size.
* For "fan out reads of one signal at a time across many shots on S3" workloads
  specifically, Zarr v3 with sharding is the better choice. For the same access
  pattern on local storage, HDF5 still wins decisively.
* For both formats, the larger `m7i.48xlarge` instance is well justified for
  shots-mode workloads (near-linear scaling down to single-second runtimes for
  HDF5 and to ~34 s for Zarr on S3) but its added cost is harder to justify for
  signals-mode workloads on local storage, where the gain over the `m5.4xlarge`
  is modest. For signals-mode workloads on S3, the larger instance helps both
  formats but neither reaches the runtimes of the shots-mode pattern.

---


## Benchmark Analysis

In [ ]:
import itertools
import pandas as pd
import hvplot.pandas  # noqa: F401
import holoviews as hv

hv.extension("bokeh")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

Read benchmark data for the `m5.4xlarge` EC2 instance:

In [ ]:
bench_data = pd.concat(
    [
        pd.read_csv("./ec2-local-v9_combined_23_disrupt.csv").assign(where="local"),
        pd.read_csv("./ec2-s3-v9_combined_23_disrupt.csv").assign(where="S3"),
    ],
    ignore_index=True,
)
bench_data.info(verbose=True)

Replace:
* Page cache size numbers with human friendly values.

In [ ]:
def fix_page_cache(df: pd.DataFrame) -> None:
    df.replace(
        {
            "pb-size": {0: "off", 268435456: "256MiB", 70000000: "70MB"},
        },
        inplace=True,
    )

In [ ]:
fix_page_cache(bench_data)

Page cache sizes in bytes used in the benchmarks:

In [ ]:
bench_data["pb-size"].unique()

The benchmark parameter combinations:

In [ ]:
bench_data[["obj-type", "pb-size", "num-workers"]].drop_duplicates()

### Total Runtime and Peformance

Total benchmark runtime in the `total-runtime` column is the elapsed time of the entire benchmark as measured by the main process. The total runtime encompasses:
1. Dividing data access job across Dask workers and their intialization.
1. All Dask workers completing their jobs.
1. Collecting Dask worker benchmark data.

Performance improvements with more Dask workers can be evaluated by computing speed-up ratios from the _baseline_ benchmark. The baseline benchmark is always the one with the smallest number of Dask workers for all other benchmark parameters being fixed. For local files, the baseline is always the benchmark without page buffering (page buffer is off).

In [ ]:
def speedup(df: pd.DataFrame) -> pd.DataFrame:
    """Add speedup to a new DataFrame."""
    # Create the base output DataFrame...
    out = df[
        ["obj-type", "pb-size", "num-workers", "total-runtime", "where"]
    ].drop_duplicates(ignore_index=True)

    # Initialize the new column for speedup...
    out["speedup"] = 0.0

    where = out["where"].unique()
    obj_type = out["obj-type"].unique()
    for obj, where in itertools.product(obj_type, where):
        mask = (out["where"] == where) & (out["obj-type"] == obj)
        if where == "local":
            baseline_idx = out.loc[
                mask & (out["pb-size"] == "off"), "num-workers"
            ].idxmin()
        else:
            baseline_idx = out.loc[mask, "num-workers"].idxmin()
        baseline_time = out.loc[baseline_idx, "total-runtime"]
        out.loc[mask, "speedup"] = baseline_time / out.loc[mask, "total-runtime"]

    return out

In [ ]:
perf_data = speedup(bench_data)

#### Reading All Signals from All Shot Files: Per Signal

Total runtime and speed-up ratios for reading all signals, one after another, from all shot files either in local file system or in S3:

In [ ]:
data = perf_data.query("where == 'local' and `obj-type` == 'signals'")
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Speed-up for local files with and without page caches (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)

In [ ]:
data = perf_data.query("where == 'S3' and `obj-type` == 'signals'")
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for S3 files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Speed-up for S3 files (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)

In [ ]:
lc_grp = perf_data.query("where == 'local' and `obj-type` == 'signals'").groupby(
    ["pb-size", "num-workers"]
)
s3_grp = perf_data.query("where == 'S3' and `obj-type` == 'signals'").groupby(
    ["pb-size", "num-workers"]
)
(
    lc_grp["total-runtime"]
    .first()
    .loc["70MB"]
    .hvplot.scatter(label="Local files, page buffer 70 MB")
    * lc_grp["total-runtime"]
    .first()
    .loc["off"]
    .hvplot.scatter(label="Local files, no page buffering")
    * s3_grp["total-runtime"]
    .first()
    .loc["70MB"]
    .hvplot.scatter(label="S3 files, page buffer 70 MB")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local and S3 files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    ylim=(10, None),
    height=400,
    width=500,
    logy=True,
)

#### Reading All Signals from All Shot Files: Per File

In [ ]:
obj_type = "shots"

In [ ]:
data = perf_data.query("where == 'local' and `obj-type` == @obj_type")

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Speed-up for local files with and without page caches (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)

In [ ]:
data = perf_data.query("where == 'S3' and `obj-type` == @obj_type")

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for S3 files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Speed-up for S3 files (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)

In [ ]:
lc_grp = perf_data.query("where == 'local' and `obj-type` == @obj_type").groupby(
    ["pb-size", "num-workers"]
)
s3_grp = perf_data.query("where == 'S3' and `obj-type` == @obj_type").groupby(
    ["pb-size", "num-workers"]
)
(
    lc_grp["total-runtime"]
    .first()
    .loc["70MB"]
    .hvplot.scatter(label="Local files, page buffer 70 MB")
    * lc_grp["total-runtime"]
    .first()
    .loc["off"]
    .hvplot.scatter(label="Local files, no page buffering")
    * s3_grp["total-runtime"]
    .first()
    .loc["70MB"]
    .hvplot.scatter(label="S3 files, page buffer 70 MB")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local and S3 files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    ylim=(1, None),
    height=400,
    width=500,
    logy=True,
)

### Worker Open File Times

Each Dask worker had a number of HDF5 files to open in order to read data from them. The time taken to open each file was recorded and their median value per worker was included in the benchmark run results.

In [ ]:
bench_data[bench_data["where"] == "local"].hvplot.box(
    y="median-open-file-time",
    by=["num-workers", "pb-size"],
).options(
    title="Local Files",
    height=400,
    show_legend=False,
    xlabel="File page buffer size, Number of Dask workers",
    ylabel="Worker median open file time / [seconds]",
    show_grid=True,
)

In [ ]:
bench_data[bench_data["where"] == "S3"].hvplot.box(
    y="median-open-file-time",
    by=["num-workers", "pb-size"],
).options(
    title="S3 Files",
    height=400,
    show_legend=False,
    xlabel="File page buffer size, Number of Dask workers",
    ylabel="Worker median open file time / [seconds]",
    show_grid=True,
)

### Impact of Opening Files on Total Runtime

Each Dask worker must open an HDF5 file prior to reading a signal from it. Repeated opening of files with unchanged content effectively incurs a time overhead on total runtime. Each Dask worker reports the median time per one HDF5 file open (column `median-open-file-time`) and the number of files opened (`num-open-files`). The data from these two columns can be used to estimate what is the aggregate impact of opening all the files on total runtime.

We define _Aggregated File Open Overhead (AFOO) Index_ as:

$$\frac{ \sum_{i=0}^{N} t_{i}}{N \cdot T}$$

where:
* $t_{i}$ total file open time of one worker
* $N$ number of Dask workers
* $T$ total runtime

The AFOO index has a value between 0 and 1, with larger values indicating higher impact of opening files on runtime.

| AFOO Value | Interpretation |
| :--- | :--- |
| < 0.05 (5%) | Negligible. File opening is not a bottleneck. Optimization will yield minimal gains. |
| 0.05 - 0.20 | Moderate. There is overhead. |
| > 0.20 (20%) | Significant bottleneck. A major portion of the workers capacity is unproductive. |

#### Per Signal Reading

In [ ]:
obj_type = "signals"
data = bench_data.query("where == 'local' and  `obj-type` == @obj_type").copy()

In [ ]:
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
grp = data.groupby(["num-workers", "pb-size"])

In [ ]:
(grp["worker-open-files-time"].sum()).hvplot.bar(
    ylabel="Time / [s]", title="Total time workers spent opening local files"
)

In [ ]:
(
    grp["worker-open-files-time"].sum()
    / (grp["num-workers"].first() * grp["total-runtime"].first())
).hvplot.bar(grid=True, title="Aggregated File Open Overhead Index for Local Files")

In [ ]:
data = bench_data.query("where == 'S3' and  `obj-type` == @obj_type").copy()

In [ ]:
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
grp = data.groupby(["num-workers", "pb-size"])

In [ ]:
(
    grp["worker-open-files-time"].sum()
    / (grp["num-workers"].first() * grp["total-runtime"].first())
).hvplot.bar(grid=True, title="Aggregated File Open Overhead Index for S3 Files")

The above values show that reading from the files in S3 was significantly impacted (slowed) by just opening the files.

#### Per File Reading

In [ ]:
obj_type = "shots"
data = bench_data.query("where == 'local' and  `obj-type` == @obj_type").copy()

In [ ]:
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
grp = data.groupby(["num-workers", "pb-size"])

In [ ]:
(grp["worker-open-files-time"].sum()).hvplot.bar(
    ylabel="Time / [s]", title="Total time workers spent opening local files"
)

In [ ]:
(
    grp["worker-open-files-time"].sum()
    / (grp["num-workers"].first() * grp["total-runtime"].first())
).hvplot.bar(grid=True, title="Aggregated File Open Overhead Index for Local Files")

In [ ]:
data = bench_data.query("where == 'S3' and  `obj-type` == @obj_type").copy()

In [ ]:
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
grp = data.groupby(["num-workers", "pb-size"])

In [ ]:
(
    grp["worker-open-files-time"].sum()
    / (grp["num-workers"].first() * grp["total-runtime"].first())
).hvplot.bar(grid=True, title="Aggregated File Open Overhead Index for S3 Files")

## One Signal Read Time Estimate

Which _object_ is read depends on the `obj-type` column:

In [ ]:
bench_data["obj-type"].unique()

The shot data was read either per signal (`obj-type="signals"`) or per file
(`obj-type="shots"`) from all the files. Each worker reported the total time
spent reading just the data (HDF5 datasets). How to interpret these values
depends on `obj-type`.

* **`obj-type`="signals"**: The worker's value is for reading the same
  signal from a number of files. This enables deriving accurate estimate for
  that signal.

    1. Compute the average read time for a specific signal per file per worker
       by dividing `read-data-time` with `num-objs`.
    1. Group these worker values by the signal (column `obj-id`).
    1. Compute the 75th percentile for every signal group.
    1. Compute the 75th percentile from all signal groups' 75th percentiles.

* **`obj-type`="shots"**: The worker's value is for reading all the signals
  found in a number of files. It can only be used to derive an approximate
  estimate for _any_ signal.

    1. Compute the average read time for any signal per file per worker by
       dividing `read-data-time` with `num-objs`.
    1. Compute the 75th percentile from all the averaged values.

In [ ]:
data = bench_data.query(
    "where == 'local' and `pb-size` == 'off' and `obj-type` == 'signals'"
)
est_local_read_signal_time_per_signal = (
    (data["read-data-time"] / data["num-objs"])
    .groupby(data["obj-id"])
    .quantile(0.75)
    .quantile(0.75)
)
est_local_read_signal_time_per_signal

In [ ]:
data = bench_data.query("where == 'S3' and `obj-type` == 'signals'")
est_s3_read_signal_time_per_signal = (
    (data["read-data-time"] / data["num-objs"])
    .groupby(data["obj-id"])
    .quantile(0.75)
    .quantile(0.75)
)
est_s3_read_signal_time_per_signal

The large difference in the computed estimates (~0.001 seconds for local files, ~0.21 seconds for S3 files) is caused by slower reading of data from S3. Using the 75th percentile favored the slower signal read times which likely reflect slower reading of new (not cached) file pages from the S3 files.

Now estimates for the per file data:

In [ ]:
data = bench_data.query(
    "where == 'local' and `pb-size` == 'off' and `obj-type` == 'shots'"
)
est_local_read_signal_time_per_file = (
    data["read-data-time"] / data["num-objs"]
).quantile(0.75)
est_local_read_signal_time_per_file

In [ ]:
data = bench_data.query("where == 'S3' and `obj-type` == 'shots'")
est_s3_read_signal_time_per_file = (data["read-data-time"] / data["num-objs"]).quantile(
    0.75
)
est_s3_read_signal_time_per_file

## Total Runtime Predictions

With estimated read time of one signal per file and the measured median open file times, we can predict how much time it would take to read $N$ ($N>0$) signals from $M$ ($M>0$) HDF5 files.

In [ ]:
est_lc_open_one_file_time = bench_data.query("where == 'local' and `pb-size` == 'off'")[
    "median-open-file-time"
].quantile(0.75)
est_lc_open_one_file_time

In [ ]:
est_s3_open_one_file_time = bench_data.query("where == 'S3'")[
    "median-open-file-time"
].quantile(0.75)
est_s3_open_one_file_time

#### Estimate Total Runtime Without File Opens

According to the AFOO index calculated above, reading signals from the HDF5 files in S3 is severely impacted by opening files prior to any access to that file's content. What would be new total runtimes if somehow the workers don't need to open the files at all?

The estimate below uses the measured time of a worker spent only reading a signal from its list of HDF5 files (column: `read-data-time`). These times are grouped based on the file page cache size, the total number of workers, and a specific worker. Each group represents the signals read by the same worker per one benchmark configuration. The estimated total runtime is based on the longest reported read data time by a single worker because all the workers execute in parallel and, hence, the slowest worker defines what the total runtime would be.

est_tot_runtime = (
    s3_data.groupby(["pb-size", "num-workers", "worker#"])["read-data-time"]
    .sum()
    .groupby(level=["pb-size", "num-workers"])
    .max()
)

(
    lc_grp["total-runtime"]
    .first()
    .loc["off"]
    .hvplot.scatter(label="Local files, no page buffering")
    * est_tot_runtime.loc["256MiB"].hvplot.scatter(
        label="Estimate for S3 files without file opens"
    )
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime comparison",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, lc["num-workers"].max() + 1),
    ylim=(10, None),
    height=400,
    width=500,
    logy=True,
)

---

## Benchmarks on a Large EC2 Instance

To explore further the impact of system resources on parallel execution, we ran the benchmarks on an EC2 instance (`m7i.48xlarge`) with 192 vCPUs and 768 GiB memory.

In [ ]:
big_data = pd.concat(
    [
        pd.read_csv("./big-ec2-local-v9_combined_23_disrupt.csv.zip").assign(
            where="local"
        ),
        pd.read_csv("./big-ec2-s3-v9_combined_23_disrupt.csv.zip").assign(where="S3"),
    ],
    ignore_index=True,
)
fix_page_cache(big_data)
big_data = big_data.astype(
    {
        "pb-size": "category",
        "obj-id": "category",
        "obj-type": "category",
        "where": "category",
    }
)
big_data.info()

In [ ]:
actual_max_workers = big_data.groupby("num-workers")["worker#"].transform("max")
if len(big_data[actual_max_workers < big_data["num-workers"]]):
    print(
        "WARNING: Some of the benchmarks ran with less workers than expected. Assigning actual value for number of workers"
    )
    big_data["num-workers"] = actual_max_workers
del actual_max_workers

In [ ]:
big_perf = speedup(big_data)

### Per Signal

In [ ]:
obj_type = "signals"

In [ ]:
data = big_perf.query("where == 'local' and `obj-type` == @obj_type")
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / data["num-workers"].min(), y_intercept=0).opts(
        color="pink", line_width=2
    )
).options(
    show_grid=True,
    legend_position="top_right",
    title="Speed-up for local files with and without page caches (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Total runtime for local files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)

In [ ]:
data = big_perf.query("where == 'S3' and `obj-type` == @obj_type")
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / data["num-workers"].min(), y_intercept=0).opts(
        color="pink", line_width=2
    )
).options(
    show_grid=True,
    legend_position="bottom_right",
    title="Speed-up for S3 files (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for S3 files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)

### Per File

In [ ]:
obj_type = "shots"

In [ ]:
data = big_perf.query("where == 'local' and `obj-type` == @obj_type")
plot_kwargs = {
    "x": "num-workers",
    # "by": ["pb-size"],
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / data["num-workers"].min(), y_intercept=0).opts(
        color="pink", line_width=2
    )
).options(
    show_grid=True,
    legend_position="top_right",
    title="Speed-up for local files without page cache (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    # "by": ["pb-size"],
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)

In [ ]:
data = big_perf.query("where == 'S3' and `obj-type` == @obj_type")
plot_kwargs = {
    "x": "num-workers",
    # "by": ["pb-size"],
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / data["num-workers"].min(), y_intercept=0).opts(
        color="pink", line_width=2
    )
).options(
    show_grid=True,
    legend_position="bottom_right",
    title="Speed-up for S3 files (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    # xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)

In [ ]:
plot_kwargs = {
    "x": "num-workers",
    "by": ["pb-size"],
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for S3 files",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)

### AFOO

#### Per File

In [ ]:
obj_type = "shots"

In [ ]:
data = big_data.query("where == 'S3' and `obj-type` == @obj_type").copy(deep=True)
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
data["pb-size"] = data["pb-size"].astype(str)
data_grp = data.groupby(["num-workers", "pb-size"], observed=True)
(
    data_grp["worker-open-files-time"].sum()
    / (data_grp["num-workers"].first() * data_grp["total-runtime"].first())
).reset_index(name="overhead").hvplot.bar(
    x="num-workers",
    y="overhead",
    by="pb-size",
    grid=True,
    title="Aggregated File Open Overhead Index for S3 Files",
)

#### Per Signal

In [ ]:
obj_type = "signals"

In [ ]:
data = big_data.query("where == 'S3' and `obj-type` == @obj_type").copy(deep=True)
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
data["pb-size"] = data["pb-size"].astype(str)
data_grp = data.groupby(["num-workers", "pb-size"], observed=True)
(
    data_grp["worker-open-files-time"].sum()
    / (data_grp["num-workers"].first() * data_grp["total-runtime"].first())
).reset_index(name="overhead").hvplot.bar(
    x="num-workers",
    y="overhead",
    by="pb-size",
    grid=True,
    title="Aggregated File Open Overhead Index for S3 Files",
)

---

# Analysis of Benchmark Results for the Zarr v3 Equivalents of the v9_combined_23_disrupt Stores

This section repeats the same benchmark analysis as above for Zarr stores
produced by converting the 100 single-shot HDF5 files. The Zarr stores have
the following properties:

* One self-contained Zarr v3 store per shot. The on-disk store is a directory
  (locally) or an S3 prefix containing the Zarr v3 metadata, consolidated
  metadata, and one object per shard.
* The Zarr group hierarchy mirrors the HDF5 file: `/shots/<SHOT_ID>/signals/
  <signal MDSplus expression>`, with the dimension coordinates of each signal
  recorded via the Xarray-compatible `_ARRAY_DIMENSIONS` attribute on the
  signal arrays.
* Per-array chunk shapes and GZip compression were carried over from the
  source HDF5 datasets. With sharding enabled, for arrays whose source HDF5
  dataset contained more than one chunk, all chunks of the Zarr array are
  bundled into a single shard. Consolidated metadata is generated at the end
  of the conversion so opening a store does not require listing every array.
* Variable-length string datasets are converted using the standard Zarr v3
  string handling (no `numcodecs.VLenUTF8`).

The benchmarks were run on the same two AWS EC2 instances as the HDF5
benchmarks:

* `m5.4xlarge` with 16 vCPUs, 64 GiB memory, up to 10 Gbps network bandwidth.
* `m7i.48xlarge` with 192 vCPUs, 768 GiB memory, 50 Gbps network bandwidth,
  40 Gbps EBS bandwidth.

The benchmark cases were created by combining the following parameters:

* 100 Zarr stores.
* The stores were placed either in the local file system (EBS) or in an S3
  bucket. Both are accessed through `zarr.storage.ObjectStore` wrapping an
  `obstore` `LocalStore` or `S3Store`, respectively.
* Data reading tasks were shared among a range of Dask workers, never
  exceeding one worker per vCPU. The actual number of workers in each
  benchmark case was adjusted, when necessary, so that no worker was idle.
* The HDF5 file page cache axis from the HDF5 analysis is not applicable in
  this setup; the only remaining benchmark parameters are the location of the
  stores (`local` vs `S3`), the access pattern (`obj-type`), and the number
  of Dask workers.
* Read all signals from all stores using two access patterns:
  * One signal at a time from all stores. Each Dask task reads one signal
    from a subset of the stores. Recorded as `obj-type="signals"`.
  * All signals from one store at a time. Each Dask task reads all signals
    from a subset of the stores. Recorded as `obj-type="shots"`.


## Zarr Benchmark Analysis

Read benchmark data for the `m5.4xlarge` EC2 instance:

In [ ]:
zarr_bench_data = pd.concat(
    [
        pd.read_csv("./zarr-ec2-local-v9_combined_23_disrupt.csv").assign(
            where="local"
        ),
        pd.read_csv("./zarr-ec2-s3-v9_combined_23_disrupt.csv").assign(where="S3"),
    ],
    ignore_index=True,
)
zarr_bench_data.info(verbose=True)


The benchmark parameter combinations:

In [ ]:
zarr_bench_data[["obj-type", "num-workers"]].drop_duplicates()


### Total Runtime and Performance

Total benchmark runtime in the `total-runtime` column is the elapsed time of
the entire benchmark as measured by the main process, identical to the HDF5
case. The total runtime encompasses:
1. Dividing data access job across Dask workers and their initialization.
1. All Dask workers completing their jobs.
1. Collecting Dask worker benchmark data.

Performance improvements with more Dask workers are evaluated by computing
speed-up ratios from the _baseline_ benchmark, defined as the benchmark with
the smallest number of Dask workers for the given access pattern and store
location. There is no `pb-size` axis here, so the baseline is unique for each
(`where`, `obj-type`) pair.


In [ ]:
def zarr_speedup(df: pd.DataFrame) -> pd.DataFrame:
    """Add speedup to a new DataFrame for the Zarr benchmarks (no pb-size)."""
    out = df[["obj-type", "num-workers", "total-runtime", "where"]].drop_duplicates(
        ignore_index=True
    )

    # Initialize the new column for speedup...
    out["speedup"] = 0.0

    where_vals = out["where"].unique()
    obj_types = out["obj-type"].unique()
    for obj, where in itertools.product(obj_types, where_vals):
        mask = (out["where"] == where) & (out["obj-type"] == obj)
        baseline_idx = out.loc[mask, "num-workers"].idxmin()
        baseline_time = out.loc[baseline_idx, "total-runtime"]
        out.loc[mask, "speedup"] = baseline_time / out.loc[mask, "total-runtime"]

    return out


In [ ]:
zarr_perf_data = zarr_speedup(zarr_bench_data)


#### Reading All Signals from All Shot Stores: Per Signal

Total runtime and speed-up ratios for reading all signals, one after another,
from all shot stores either in the local file system or in S3:


In [ ]:
data = zarr_perf_data.query("where == 'local' and `obj-type` == 'signals'")
plot_kwargs = {
    "x": "num-workers",
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Speed-up for local Zarr stores (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


In [ ]:
data = zarr_perf_data.query("where == 'S3' and `obj-type` == 'signals'")
plot_kwargs = {
    "x": "num-workers",
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for S3 Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Speed-up for S3 Zarr stores (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


In [ ]:
lc_grp = zarr_perf_data.query("where == 'local' and `obj-type` == 'signals'").groupby(
    ["num-workers"]
)
s3_grp = zarr_perf_data.query("where == 'S3' and `obj-type` == 'signals'").groupby(
    ["num-workers"]
)
(
    lc_grp["total-runtime"].first().hvplot.scatter(label="Local Zarr stores")
    * s3_grp["total-runtime"].first().hvplot.scatter(label="S3 Zarr stores")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local and S3 Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    ylim=(10, None),
    height=400,
    width=500,
    logy=True,
)


#### Reading All Signals from All Shot Stores: Per Store

In [ ]:
obj_type = "shots"


In [ ]:
data = zarr_perf_data.query("where == 'local' and `obj-type` == @obj_type")


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Speed-up for local Zarr stores (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


In [ ]:
data = zarr_perf_data.query("where == 'S3' and `obj-type` == @obj_type")


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for S3 Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Speed-up for S3 Zarr stores (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


In [ ]:
lc_grp = zarr_perf_data.query("where == 'local' and `obj-type` == @obj_type").groupby(
    ["num-workers"]
)
s3_grp = zarr_perf_data.query("where == 'S3' and `obj-type` == @obj_type").groupby(
    ["num-workers"]
)
(
    lc_grp["total-runtime"].first().hvplot.scatter(label="Local Zarr stores")
    * s3_grp["total-runtime"].first().hvplot.scatter(label="S3 Zarr stores")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local and S3 Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    ylim=(1, None),
    height=400,
    width=500,
    logy=True,
)


### Worker Open Store Times

Each Dask worker had a number of Zarr stores to open in order to read data
from them. Opening a Zarr store with consolidated metadata involves fetching
and parsing the consolidated metadata document. The time taken to open each
store was recorded and the median value per worker was included in the
benchmark run results.


In [ ]:
zarr_bench_data[zarr_bench_data["where"] == "local"].hvplot.box(
    y="median-open-file-time",
    by=["num-workers"],
).options(
    title="Local Zarr stores",
    height=400,
    show_legend=False,
    xlabel="Number of Dask workers",
    ylabel="Worker median open store time / [seconds]",
    show_grid=True,
)


In [ ]:
zarr_bench_data[zarr_bench_data["where"] == "S3"].hvplot.box(
    y="median-open-file-time",
    by=["num-workers"],
).options(
    title="S3 Zarr stores",
    height=400,
    show_legend=False,
    xlabel="Number of Dask workers",
    ylabel="Worker median open store time / [seconds]",
    show_grid=True,
)


### Impact of Opening Stores on Total Runtime

The same _Aggregated File Open Overhead (AFOO) Index_ defined for the HDF5
analysis is reused here, with stores in place of files:

$$\frac{ \sum_{i=0}^{N} t_{i}}{N \cdot T}$$

where:
* $t_{i}$ total store open time of one worker
* $N$ number of Dask workers
* $T$ total runtime

The AFOO index has a value between 0 and 1, with larger values indicating
higher impact of opening stores on runtime.

| AFOO Value | Interpretation |
| :--- | :--- |
| < 0.05 (5%) | Negligible. Store opening is not a bottleneck. Optimization will yield minimal gains. |
| 0.05 - 0.20 | Moderate. There is overhead. |
| > 0.20 (20%) | Significant bottleneck. A major portion of the workers capacity is unproductive. |


#### Per Signal Reading

In [ ]:
obj_type = "signals"
data = zarr_bench_data.query("where == 'local' and  `obj-type` == @obj_type").copy()


In [ ]:
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
grp = data.groupby(["num-workers"])


In [ ]:
(grp["worker-open-files-time"].sum()).hvplot.bar(
    ylabel="Time / [s]",
    title="Total time workers spent opening local Zarr stores",
)


In [ ]:
(
    grp["worker-open-files-time"].sum()
    / (grp["num-workers"].first() * grp["total-runtime"].first())
).hvplot.bar(
    grid=True,
    title="Aggregated File Open Overhead Index for Local Zarr Stores",
)


In [ ]:
data = zarr_bench_data.query("where == 'S3' and  `obj-type` == @obj_type").copy()


In [ ]:
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
grp = data.groupby(["num-workers"])


In [ ]:
(
    grp["worker-open-files-time"].sum()
    / (grp["num-workers"].first() * grp["total-runtime"].first())
).hvplot.bar(
    grid=True,
    title="Aggregated File Open Overhead Index for S3 Zarr Stores",
)


The above values show that, for the per-signal access pattern, opening the
Zarr stores accounts for a substantial fraction of the workers' time on both
local and S3 storage. The S3 case is the more severe of the two given the
larger absolute open time per store. The dominant cause is that, in this
access pattern, every worker has to reopen every store in its assigned
batch once for each of the signals it processes; the number of opens grows
with both the number of stores and the number of signals.


#### Per Store Reading

In [ ]:
obj_type = "shots"
data = zarr_bench_data.query("where == 'local' and  `obj-type` == @obj_type").copy()


In [ ]:
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
grp = data.groupby(["num-workers"])


In [ ]:
(grp["worker-open-files-time"].sum()).hvplot.bar(
    ylabel="Time / [s]",
    title="Total time workers spent opening local Zarr stores",
)


In [ ]:
(
    grp["worker-open-files-time"].sum()
    / (grp["num-workers"].first() * grp["total-runtime"].first())
).hvplot.bar(
    grid=True,
    title="Aggregated File Open Overhead Index for Local Zarr Stores",
)


In [ ]:
data = zarr_bench_data.query("where == 'S3' and  `obj-type` == @obj_type").copy()


In [ ]:
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
grp = data.groupby(["num-workers"])


In [ ]:
(
    grp["worker-open-files-time"].sum()
    / (grp["num-workers"].first() * grp["total-runtime"].first())
).hvplot.bar(
    grid=True,
    title="Aggregated File Open Overhead Index for S3 Zarr Stores",
)


## One Signal Read Time Estimate (Zarr)

The same per-signal read time estimation procedure used in the HDF5 analysis
is applied to the Zarr benchmarks. The two `obj-type` values mean the same
thing as before:


In [ ]:
zarr_bench_data["obj-type"].unique()


The shot data was read either per signal (`obj-type="signals"`) or per store
(`obj-type="shots"`) from all the Zarr stores. Each worker reported the total
time spent reading just the data (Zarr arrays). How to interpret these values
depends on `obj-type`.

* **`obj-type`="signals"**: The worker's value is for reading the same
  signal from a number of Zarr stores. This enables deriving an accurate
  estimate for that signal.

    1. Compute the average read time for a specific signal per store per
       worker by dividing `read-data-time` with `num-objs`.
    1. Group these worker values by the signal (column `obj-id`).
    1. Compute the 75th percentile for every signal group.
    1. Compute the 75th percentile from all signal groups' 75th percentiles.

* **`obj-type`="shots"**: The worker's value is for reading all the signals
  found in a number of Zarr stores. It can only be used to derive an
  approximate estimate for _any_ signal.

    1. Compute the average read time for any signal per store per worker by
       dividing `read-data-time` with `num-objs`.
    1. Compute the 75th percentile from all the averaged values.


In [ ]:
data = zarr_bench_data.query("where == 'local' and `obj-type` == 'signals'")
zarr_est_local_read_signal_time_per_signal = (
    (data["read-data-time"] / data["num-objs"])
    .groupby(data["obj-id"])
    .quantile(0.75)
    .quantile(0.75)
)
zarr_est_local_read_signal_time_per_signal


In [ ]:
data = zarr_bench_data.query("where == 'S3' and `obj-type` == 'signals'")
zarr_est_s3_read_signal_time_per_signal = (
    (data["read-data-time"] / data["num-objs"])
    .groupby(data["obj-id"])
    .quantile(0.75)
    .quantile(0.75)
)
zarr_est_s3_read_signal_time_per_signal


The estimated per-store read times below describe how long it takes a worker
to read all signals from a single Zarr store, on average.


In [ ]:
data = zarr_bench_data.query("where == 'local' and `obj-type` == 'shots'")
zarr_est_local_read_signal_time_per_store = (
    data["read-data-time"] / data["num-objs"]
).quantile(0.75)
zarr_est_local_read_signal_time_per_store


In [ ]:
data = zarr_bench_data.query("where == 'S3' and `obj-type` == 'shots'")
zarr_est_s3_read_signal_time_per_store = (
    data["read-data-time"] / data["num-objs"]
).quantile(0.75)
zarr_est_s3_read_signal_time_per_store


## Total Runtime Predictions (Zarr)

With estimated read time of one signal per store and the measured median
open store times, we can predict how much time it would take to read $N$
($N>0$) signals from $M$ ($M>0$) Zarr stores in this setup.


In [ ]:
zarr_est_lc_open_one_store_time = zarr_bench_data.query("where == 'local'")[
    "median-open-file-time"
].quantile(0.75)
zarr_est_lc_open_one_store_time


In [ ]:
zarr_est_s3_open_one_store_time = zarr_bench_data.query("where == 'S3'")[
    "median-open-file-time"
].quantile(0.75)
zarr_est_s3_open_one_store_time


---

## Zarr Benchmarks on a Large EC2 Instance

To explore further the impact of system resources on parallel execution, we
ran the Zarr benchmarks on the same `m7i.48xlarge` EC2 instance with 192
vCPUs and 768 GiB memory. The maximum benchmarked worker count ended at
100 due to the total number of Zarr stores (100 stores; min. case 1 store/worker).
The benchmark CSV files are stored as ZIP archives.


In [ ]:
zarr_big_data = pd.concat(
    [
        pd.read_csv("./zarr-big-ec2-local-v9_combined_23_disrupt.csv.zip").assign(
            where="local"
        ),
        pd.read_csv("./zarr-big-ec2-s3-v9_combined_23_disrupt.csv.zip").assign(
            where="S3"
        ),
    ],
    ignore_index=True,
)
zarr_big_data = zarr_big_data.astype(
    {
        "obj-id": "category",
        "obj-type": "category",
        "where": "category",
    }
)
zarr_big_data.info()


In [ ]:
actual_max_workers = zarr_big_data.groupby("num-workers")["worker#"].transform("max")
if len(zarr_big_data[actual_max_workers < zarr_big_data["num-workers"]]):
    print(
        "WARNING: Some of the benchmarks ran with less workers than expected. Assigning actual value for number of workers"
    )
    zarr_big_data["num-workers"] = actual_max_workers
del actual_max_workers


In [ ]:
zarr_big_perf = zarr_speedup(zarr_big_data)


### Per Signal

In [ ]:
obj_type = "signals"


In [ ]:
data = zarr_big_perf.query("where == 'local' and `obj-type` == @obj_type")
plot_kwargs = {
    "x": "num-workers",
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / data["num-workers"].min(), y_intercept=0).opts(
        color="pink", line_width=2
    )
).options(
    show_grid=True,
    legend_position="top_right",
    title="Speed-up for local Zarr stores (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Total runtime for local Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


In [ ]:
data = zarr_big_perf.query("where == 'S3' and `obj-type` == @obj_type")
plot_kwargs = {
    "x": "num-workers",
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / data["num-workers"].min(), y_intercept=0).opts(
        color="pink", line_width=2
    )
).options(
    show_grid=True,
    legend_position="bottom_right",
    title="Speed-up for S3 Zarr stores (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for S3 Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


### Per Store

In [ ]:
obj_type = "shots"


In [ ]:
data = zarr_big_perf.query("where == 'local' and `obj-type` == @obj_type")
plot_kwargs = {
    "x": "num-workers",
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / data["num-workers"].min(), y_intercept=0).opts(
        color="pink", line_width=2
    )
).options(
    show_grid=True,
    legend_position="top_right",
    title="Speed-up for local Zarr stores (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for local Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


In [ ]:
data = zarr_big_perf.query("where == 'S3' and `obj-type` == @obj_type")
plot_kwargs = {
    "x": "num-workers",
}
y = "speedup"
(
    data.hvplot.line(y=y, **plot_kwargs)
    * data.hvplot.scatter(y=y, **plot_kwargs)
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / data["num-workers"].min(), y_intercept=0).opts(
        color="pink", line_width=2
    )
).options(
    show_grid=True,
    legend_position="bottom_right",
    title="Speed-up for S3 Zarr stores (>1 is better)",
    xlabel="Number of Dask workers",
    ylabel="Speed-Up",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


In [ ]:
plot_kwargs = {
    "x": "num-workers",
}
y = "total-runtime"
(
    data.hvplot.line(y=y, **plot_kwargs) * data.hvplot.scatter(y=y, **plot_kwargs)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime for S3 Zarr stores",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(data["num-workers"].min() - 1, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


### AFOO

#### Per Store

In [ ]:
obj_type = "shots"


In [ ]:
data = zarr_big_data.query("where == 'S3' and `obj-type` == @obj_type").copy(deep=True)
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
data_grp = data.groupby(["num-workers"], observed=True)
(
    data_grp["worker-open-files-time"].sum()
    / (data_grp["num-workers"].first() * data_grp["total-runtime"].first())
).reset_index(name="overhead").hvplot.bar(
    x="num-workers",
    y="overhead",
    grid=True,
    title="Aggregated File Open Overhead Index for S3 Zarr Stores",
)


#### Per Signal

In [ ]:
obj_type = "signals"


In [ ]:
data = zarr_big_data.query("where == 'S3' and `obj-type` == @obj_type").copy(deep=True)
data["worker-open-files-time"] = data["num-open-files"] * data["median-open-file-time"]
data_grp = data.groupby(["num-workers"], observed=True)
(
    data_grp["worker-open-files-time"].sum()
    / (data_grp["num-workers"].first() * data_grp["total-runtime"].first())
).reset_index(name="overhead").hvplot.bar(
    x="num-workers",
    y="overhead",
    grid=True,
    title="Aggregated File Open Overhead Index for S3 Zarr Stores",
)


---

# HDF5 vs Zarr Comparison

This section places the HDF5 and Zarr benchmark results side by side. For each
EC2 instance, comparisons are made for both access patterns (`obj-type` =
`signals` and `obj-type` = `shots`) and both storage locations (local and S3).

Slicing rules used throughout this section:

* HDF5, `local`: rows with `pb-size == 'off'` (no page-cache effect).
* HDF5, `S3`:    rows with `pb-size == '70MB'`.
* Zarr:          no `pb-size` axis.

Speed-up ratios are normalised to the HDF5 1-worker runtime in the same
`(where, obj-type)` slice, so HDF5 at 1 worker is exactly 1.0 and the Zarr
curve is directly comparable to the HDF5 multi-worker curve. For the
m7i.48xlarge S3 panels, the HDF5 benchmark started at 12 workers, so the
smallest available HDF5 worker count is used in place of 1 worker; this is
called out in the relevant cells.


## Comparison on the `m5.4xlarge` EC2 instance

`perf_data` (HDF5, m5.4xlarge) has rows for each (`where`, `obj-type`, `pb-size`,
`num-workers`) combination. We keep only the slices that are meaningful for the
comparison, drop `pb-size`, and tag the format. `zarr_perf_data` already has
the right shape (no `pb-size` column) and is tagged "Zarr".

In [ ]:
hdf5_small_cmp = pd.concat(
    [
        # Local: pb-size 'off' — closest analogue to Zarr (no page cache).
        perf_data.query("where == 'local' and `pb-size` == 'off'").assign(
            format="HDF5"
        ),
        # S3: pb-size '70MB' — usable performance.
        perf_data.query("where == 'S3' and `pb-size` == '70MB'").assign(format="HDF5"),
    ],
    ignore_index=True,
)[["where", "obj-type", "num-workers", "total-runtime", "format"]]

zarr_small_cmp = zarr_perf_data.assign(format="Zarr")[
    ["where", "obj-type", "num-workers", "total-runtime", "format"]
]

cmp_small = pd.concat([hdf5_small_cmp, zarr_small_cmp], ignore_index=True)

HDF5 1-worker runtime per (where, obj-type) for the small EC2; these are
the divisors used for the speed-up plots below.

In [ ]:
hdf5_small_1w = hdf5_small_cmp.query("`num-workers` == 1").set_index(
    ["where", "obj-type"]
)["total-runtime"]
hdf5_small_1w

### Total runtime: HDF5 vs Zarr (m5.4xlarge, local)

Local signals: HDF5 (`pb-size='off'`) vs Zarr.

In [ ]:
data = cmp_small.query("where == 'local' and `obj-type` == 'signals'")
(
    data.hvplot.line(x="num-workers", y="total-runtime", by="format")
    * data.hvplot.scatter(x="num-workers", y="total-runtime", by="format")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime, local, per-signal mode",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


Local shots: HDF5 (pb-size='off') vs Zarr.

In [ ]:
data = cmp_small.query("where == 'local' and `obj-type` == 'shots'")
(
    data.hvplot.line(x="num-workers", y="total-runtime", by="format")
    * data.hvplot.scatter(x="num-workers", y="total-runtime", by="format")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime, local, per-store mode",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


### Total runtime: HDF5 vs Zarr (m5.4xlarge, S3)

S3 signals: HDF5 (`pb-size='70MB'`) vs Zarr.

In [ ]:
data = cmp_small.query("where == 'S3' and `obj-type` == 'signals'")
(
    data.hvplot.line(x="num-workers", y="total-runtime", by="format")
    * data.hvplot.scatter(x="num-workers", y="total-runtime", by="format")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime, S3, per-signal mode",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


S3 shots: HDF5 (`pb-size='70MB'`) vs Zarr.

In [ ]:
data = cmp_small.query("where == 'S3' and `obj-type` == 'shots'")
(
    data.hvplot.line(x="num-workers", y="total-runtime", by="format")
    * data.hvplot.scatter(x="num-workers", y="total-runtime", by="format")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime, S3, per-store mode",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


### Speed-up: HDF5 vs Zarr (`m5.4xlarge`)

Speed-up = HDF5 1-worker runtime / current runtime, evaluated separately per
`(where, obj-type)` slice. By construction the HDF5 curve passes through
`(num-workers=1, speedup=1.0)`. Values larger than 1 indicate runtime smaller
than HDF5 at 1 worker; values larger than `num-workers` indicate
super-linear improvement against the HDF5 baseline.

Local signals: speed-up vs HDF5 1-worker (local, signals, `pb-size='off'`).


In [ ]:
data = cmp_small.query("where == 'local' and `obj-type` == 'signals'").copy()
baseline = hdf5_small_1w.loc[("local", "signals")]
data["speedup-vs-hdf5-1w"] = baseline / data["total-runtime"]
(
    data.hvplot.line(x="num-workers", y="speedup-vs-hdf5-1w", by="format")
    * data.hvplot.scatter(x="num-workers", y="speedup-vs-hdf5-1w", by="format")
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Speed-up vs HDF5 1 worker, local, per-signal mode",
    xlabel="Number of Dask workers",
    ylabel="Speed-up (>1 better than HDF5 at 1 worker)",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


Local shots: speed-up vs HDF5 1-worker (local, shots, `pb-size='off'`).

In [ ]:
data = cmp_small.query("where == 'local' and `obj-type` == 'shots'").copy()
baseline = hdf5_small_1w.loc[("local", "shots")]
data["speedup-vs-hdf5-1w"] = baseline / data["total-runtime"]
(
    data.hvplot.line(x="num-workers", y="speedup-vs-hdf5-1w", by="format")
    * data.hvplot.scatter(x="num-workers", y="speedup-vs-hdf5-1w", by="format")
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_right",
    title="Speed-up vs HDF5 1 worker, local, per-store mode",
    xlabel="Number of Dask workers",
    ylabel="Speed-up (>1 better than HDF5 at 1 worker)",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


S3 signals: speed-up vs HDF5 1-worker (S3, signals, `pb-size='70MB'`).

In [ ]:
data = cmp_small.query("where == 'S3' and `obj-type` == 'signals'").copy()
baseline = hdf5_small_1w.loc[("S3", "signals")]
data["speedup-vs-hdf5-1w"] = baseline / data["total-runtime"]
(
    data.hvplot.line(x="num-workers", y="speedup-vs-hdf5-1w", by="format")
    * data.hvplot.scatter(x="num-workers", y="speedup-vs-hdf5-1w", by="format")
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Speed-up vs HDF5 1 worker, S3, per-signal mode",
    xlabel="Number of Dask workers",
    ylabel="Speed-up (>1 better than HDF5 at 1 worker)",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


S3 shots: speed-up vs HDF5 1-worker (S3, shots, `pb-size='70MB'`).

In [ ]:
data = cmp_small.query("where == 'S3' and `obj-type` == 'shots'").copy()
baseline = hdf5_small_1w.loc[("S3", "shots")]
data["speedup-vs-hdf5-1w"] = baseline / data["total-runtime"]
(
    data.hvplot.line(x="num-workers", y="speedup-vs-hdf5-1w", by="format")
    * data.hvplot.scatter(x="num-workers", y="speedup-vs-hdf5-1w", by="format")
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_left",
    title="Speed-up vs HDF5 1 worker, S3, per-store mode",
    xlabel="Number of Dask workers",
    ylabel="Speed-up (>1 better than HDF5 at 1 worker)",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


### Estimated read time per signal (`m5.4xlarge`)

Both statistics already used in the per-format analysis are applied here.
Note that they have **different units** because `num-objs` plays a different
role in each mode:

* For `obj-type == 'signals'` -- units are **seconds per signal**: each
  worker's `read-data-time / num-objs` is the average time spent reading one
  signal from one file/store (here `num-objs` is the number of files/stores
  the worker visited for that signal). Those values are grouped by signal
  (`obj-id`); the 75th percentile per signal is then aggregated to its 75th
  percentile across signals (75th-of-75th).
* For `obj-type == 'shots'` -- units are **seconds per file or store**: each
  worker's `read-data-time / num-objs` is the average time spent reading all
  signals in one file/store (here `num-objs` is the number of files/stores
  the worker handled). The reported value is the 75th percentile across all
  such worker rows in the slice.

The HDF5 slices use the same `pb-size` rules as elsewhere in this section
(`off` for local, `70MB` for S3). Zarr has no `pb-size` axis. The bar chart
uses a log Y axis so the per-signal and per-store estimates are readable
together; the unit is annotated on the X-axis label of each bar group.


In [ ]:
def _est_signals_mode(df, where, pb_size=None):
    """75th-of-75th of (read-data-time / num-objs) by obj-id, for `obj-type='signals'`.
    Returns the estimated read time in **seconds per signal**."""
    q = "where == @where and `obj-type` == 'signals'"
    if pb_size is not None:
        q += " and `pb-size` == @pb_size"
    d = df.query(q)
    return (
        (d["read-data-time"] / d["num-objs"])
        .groupby(d["obj-id"], observed=True)
        .quantile(0.75)
        .quantile(0.75)
    )


def _est_shots_mode(df, where, pb_size=None):
    """75th percentile of (read-data-time / num-objs), for `obj-type='shots'`.
    Returns the estimated read time in **seconds per file (HDF5) or per store
    (Zarr)** -- i.e., the time to read all signals from a single file/store."""
    q = "where == @where and `obj-type` == 'shots'"
    if pb_size is not None:
        q += " and `pb-size` == @pb_size"
    d = df.query(q)
    return (d["read-data-time"] / d["num-objs"]).quantile(0.75)

Small-EC2 per-signal / per-store read-time estimates. The HDF5 frame is
`bench_data` (still has pb-size); the Zarr frame is `zarr_bench_data`
(no pb-size). The `unit` column makes the difference in units explicit
between the two `obj-type` rows.

In [ ]:
persig_small = pd.DataFrame(
    [
        (
            "HDF5",
            "local",
            "signals",
            "s / signal",
            _est_signals_mode(bench_data, "local", "off"),
        ),
        (
            "HDF5",
            "local",
            "shots",
            "s / file",
            _est_shots_mode(bench_data, "local", "off"),
        ),
        (
            "HDF5",
            "S3",
            "signals",
            "s / signal",
            _est_signals_mode(bench_data, "S3", "70MB"),
        ),
        ("HDF5", "S3", "shots", "s / file", _est_shots_mode(bench_data, "S3", "70MB")),
        (
            "Zarr",
            "local",
            "signals",
            "s / signal",
            _est_signals_mode(zarr_bench_data, "local"),
        ),
        (
            "Zarr",
            "local",
            "shots",
            "s / store",
            _est_shots_mode(zarr_bench_data, "local"),
        ),
        (
            "Zarr",
            "S3",
            "signals",
            "s / signal",
            _est_signals_mode(zarr_bench_data, "S3"),
        ),
        ("Zarr", "S3", "shots", "s / store", _est_shots_mode(zarr_bench_data, "S3")),
    ],
    columns=["format", "where", "obj-type", "unit", "est-read-time"],
)
persig_small

Side-by-side bar charts: HDF5 vs Zarr per scenario for the `m5.4xlarge`. The
estimates are split into two charts because the units differ between the two
modes: per signal (signals mode) and per file or per store (shots mode). The
signals-mode chart uses a log Y axis and a small Bokeh hook that anchors each
bar at the Y axis floor (Bokeh's `vbar` defaults to `bottom=0`, which is
undefined on a log axis); the shots-mode chart uses a linear Y axis.


In [ ]:
def _set_bar_log_bottom(plot, element, bottom=1e-6):
    """Bokeh hook: set every `vbar`/`hbar` glyph's `bottom` so the bars are
    anchored at the y-axis floor instead of at the default 0 (which is not
    valid on a log axis and causes bars to render against an arbitrary anchor).
    """
    p = plot.handles["plot"]
    for renderer in p.renderers:
        glyph = getattr(renderer, "glyph", None)
        if glyph is not None and hasattr(glyph, "bottom"):
            glyph.bottom = bottom


In [ ]:
psig_small = persig_small.query("`obj-type` == 'signals'")
psig_small.assign(
    scenario=(
        psig_small["where"]
        + " / "
        + psig_small["obj-type"]
        + " ["
        + psig_small["unit"]
        + "]"
    )
).hvplot.bar(
    x="scenario",
    y="est-read-time",
    by="format",
    rot=20,
).options(
    hooks=[_set_bar_log_bottom],
    show_grid=True,
    title="Estimated read time per signal -- signals mode (m5.4xlarge)",
    ylabel="Estimated read time / [s/signal]  (log scale)",
    xlabel="Scenario (where / obj-type [unit])",
    height=400,
    width=560,
    logy=True,
    ylim=(1e-6, None),
)


In [ ]:
pst_small = persig_small.query("`obj-type` == 'shots'")
pst_small.assign(
    scenario=(
        pst_small["where"]
        + " / "
        + pst_small["obj-type"]
        + " ["
        + pst_small["unit"]
        + "]"
    )
).hvplot.bar(
    x="scenario",
    y="est-read-time",
    by="format",
    rot=20,
).options(
    show_grid=True,
    title="Estimated read time per file / per store -- shots mode (m5.4xlarge)",
    ylabel="Estimated read time / [s/file or s/store]",
    xlabel="Scenario (where / obj-type [unit])",
    height=400,
    width=560,
    ylim=(0, None),
)


## Comparison on the `m7i.48xlarge` EC2 instance

`big_perf` (HDF5, m7i.48xlarge) has rows for each (`where`, `obj-type`, `pb-size`,
`num-workers`) combination.

In [ ]:
hdf5_big_cmp = pd.concat(
    [
        big_perf.query("where == 'local' and `pb-size` == 'off'").assign(format="HDF5"),
        big_perf.query("where == 'S3' and `pb-size` == '70MB'").assign(format="HDF5"),
    ],
    ignore_index=True,
)[["where", "obj-type", "num-workers", "total-runtime", "format"]]

zarr_big_cmp = zarr_big_perf.assign(format="Zarr")[
    ["where", "obj-type", "num-workers", "total-runtime", "format"]
]

cmp_big = pd.concat([hdf5_big_cmp, zarr_big_cmp], ignore_index=True)

hdf5_big_baseline = (
    hdf5_big_cmp.sort_values("num-workers")
    .groupby(["where", "obj-type"], observed=True)
    .first()
    .rename(
        columns={
            "total-runtime": "baseline-runtime",
            "num-workers": "baseline-workers",
        }
    )[["baseline-runtime", "baseline-workers"]]
)
hdf5_big_baseline

### Total runtime: HDF5 vs Zarr (m7i.48xlarge, local)

Big-EC2, local signals: HDF5 (`pb-size='off'`) vs Zarr.

In [ ]:
data = cmp_big.query("where == 'local' and `obj-type` == 'signals'")
(
    data.hvplot.line(x="num-workers", y="total-runtime", by="format")
    * data.hvplot.scatter(x="num-workers", y="total-runtime", by="format")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime, local, per-signal mode (m7i.48xlarge)",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


Big-EC2, local shots: HDF5 (`pb-size='off'`) vs Zarr.

In [ ]:
data = cmp_big.query("where == 'local' and `obj-type` == 'shots'")
(
    data.hvplot.line(x="num-workers", y="total-runtime", by="format")
    * data.hvplot.scatter(x="num-workers", y="total-runtime", by="format")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime, local, per-store mode (m7i.48xlarge)",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


### Total runtime: HDF5 vs Zarr (m7i.48xlarge, S3)

Note: the HDF5 m7i.48xlarge S3 benchmark started at 12 workers, so the HDF5
curve below is undefined for `num-workers < 12`.

Big-EC2, S3 signals: HDF5 (`pb-size='70MB'`) vs Zarr.

In [ ]:
data = cmp_big.query("where == 'S3' and `obj-type` == 'signals'")
(
    data.hvplot.line(x="num-workers", y="total-runtime", by="format")
    * data.hvplot.scatter(x="num-workers", y="total-runtime", by="format")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime, S3, per-signal mode (m7i.48xlarge)",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


Big-EC2, S3 shots: HDF5 (`pb-size='70MB'`) vs Zarr.

In [ ]:
data = cmp_big.query("where == 'S3' and `obj-type` == 'shots'")
(
    data.hvplot.line(x="num-workers", y="total-runtime", by="format")
    * data.hvplot.scatter(x="num-workers", y="total-runtime", by="format")
).options(
    show_grid=True,
    legend_position="top_right",
    title="Total runtime, S3, per-store mode (m7i.48xlarge)",
    xlabel="Number of Dask workers",
    ylabel="Runtime / [s]",
    xlim=(0, data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


### Speed-up: HDF5 vs Zarr (m7i.48xlarge)

Speed-up = HDF5 baseline runtime / current runtime, evaluated separately per
`(where, obj-type)` slice. The baseline is HDF5 at 1 worker for the local
panels and HDF5 at 12 workers for the S3 panels (because the m7i.48xlarge
HDF5 S3 benchmark did not include fewer than 12 workers). The exact baseline
worker count for each panel is taken from the `hdf5_big_baseline` table above.

For the comparison to remain apples-to-apples, every visible chart point
compares HDF5 and Zarr at the same `num-workers`: each panel restricts both
curves to `num-workers >= baseline_w`. For the local panels this is a no-op;
for the S3 panels the Zarr curve is trimmed to `num-workers >= 12`.

Big-EC2, local signals: speed-up vs HDF5 baseline (1 worker).


In [ ]:
data = cmp_big.query("where == 'local' and `obj-type` == 'signals'").copy()
baseline = hdf5_big_baseline.loc[("local", "signals"), "baseline-runtime"]
baseline_w = int(hdf5_big_baseline.loc[("local", "signals"), "baseline-workers"])
data["speedup-vs-hdf5-baseline"] = baseline / data["total-runtime"]
# Restrict both curves to num-workers >= baseline_w so each visible
# chart point compares HDF5 and Zarr at the same worker count.
data = data[data["num-workers"] >= baseline_w]
(
    data.hvplot.line(x="num-workers", y="speedup-vs-hdf5-baseline", by="format")
    * data.hvplot.scatter(x="num-workers", y="speedup-vs-hdf5-baseline", by="format")
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / baseline_w, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="top_right",
    title=f"Speed-up vs HDF5 with {baseline_w} worker(s), local, per-signal mode",
    xlabel="Number of Dask workers",
    ylabel=f"Speed-up (>1 better than HDF5 at {baseline_w} worker(s))",
    xlim=(max(0, baseline_w - 1), data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


Big-EC2, local shots: speed-up vs HDF5 baseline (1 worker).

In [ ]:
data = cmp_big.query("where == 'local' and `obj-type` == 'shots'").copy()
baseline = hdf5_big_baseline.loc[("local", "shots"), "baseline-runtime"]
baseline_w = int(hdf5_big_baseline.loc[("local", "shots"), "baseline-workers"])
data["speedup-vs-hdf5-baseline"] = baseline / data["total-runtime"]
# Restrict both curves to num-workers >= baseline_w so each visible
# chart point compares HDF5 and Zarr at the same worker count.
data = data[data["num-workers"] >= baseline_w]
(
    data.hvplot.line(x="num-workers", y="speedup-vs-hdf5-baseline", by="format")
    * data.hvplot.scatter(x="num-workers", y="speedup-vs-hdf5-baseline", by="format")
    * hv.HLine(1).opts(line_width=2, color="pink")
    * hv.Slope(slope=1 / baseline_w, y_intercept=0).opts(color="pink", line_width=2)
).options(
    show_grid=True,
    legend_position="bottom_right",
    title=f"Speed-up vs HDF5 with {baseline_w} worker(s), local, per-store mode",
    xlabel="Number of Dask workers",
    ylabel=f"Speed-up (>1 better than HDF5 at {baseline_w} worker(s))",
    xlim=(max(0, baseline_w - 1), data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=580,
)


Big-EC2, S3 signals: speed-up vs HDF5 baseline (12 workers). Both HDF5 and
Zarr curves are restricted to `num-workers >= 12` so each visible chart point
compares the two formats at the same worker count.


In [ ]:
data = cmp_big.query("where == 'S3' and `obj-type` == 'signals'").copy()
baseline = hdf5_big_baseline.loc[("S3", "signals"), "baseline-runtime"]
baseline_w = int(hdf5_big_baseline.loc[("S3", "signals"), "baseline-workers"])
data["speedup-vs-hdf5-baseline"] = baseline / data["total-runtime"]
# Restrict both curves to num-workers >= baseline_w so each visible
# chart point compares HDF5 and Zarr at the same worker count.
data = data[data["num-workers"] >= baseline_w]
(
    data.hvplot.line(x="num-workers", y="speedup-vs-hdf5-baseline", by="format")
    * data.hvplot.scatter(x="num-workers", y="speedup-vs-hdf5-baseline", by="format")
    * hv.HLine(1).opts(line_width=2, color="pink")
).options(
    show_grid=True,
    legend_position="bottom_right",
    title=f"Speed-up vs HDF5 with {baseline_w} worker(s), S3, per-signal mode",
    xlabel="Number of Dask workers",
    ylabel=f"Speed-up (>1 better than HDF5 at {baseline_w} worker(s))",
    xlim=(max(0, baseline_w - 1), data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


Big-EC2, S3 shots: speed-up vs HDF5 baseline (12 workers). Both HDF5 and
Zarr curves are restricted to `num-workers >= 12`.


In [ ]:
data = cmp_big.query("where == 'S3' and `obj-type` == 'shots'").copy()
baseline = hdf5_big_baseline.loc[("S3", "shots"), "baseline-runtime"]
baseline_w = int(hdf5_big_baseline.loc[("S3", "shots"), "baseline-workers"])
data["speedup-vs-hdf5-baseline"] = baseline / data["total-runtime"]
# Restrict both curves to num-workers >= baseline_w so each visible
# chart point compares HDF5 and Zarr at the same worker count.
data = data[data["num-workers"] >= baseline_w]
(
    data.hvplot.line(x="num-workers", y="speedup-vs-hdf5-baseline", by="format")
    * data.hvplot.scatter(x="num-workers", y="speedup-vs-hdf5-baseline", by="format")
    * hv.HLine(1).opts(line_width=2, color="pink")
).options(
    show_grid=True,
    legend_position="top_left",
    title=f"Speed-up vs HDF5 with {baseline_w} worker(s), S3, per-store mode",
    xlabel="Number of Dask workers",
    ylabel=f"Speed-up (>1 better than HDF5 at {baseline_w} worker(s))",
    xlim=(max(0, baseline_w - 1), data["num-workers"].max() + 1),
    ylim=(0, None),
    height=400,
    width=500,
)


### Estimated read time per signal (`m7i.48xlarge`)

Same statistics as for the m5.4xlarge above, applied to `big_data` (HDF5)
and `zarr_big_data` (Zarr). The HDF5 slicing rules are unchanged: `pb-size`
== 'off' for local, `pb-size` == '70MB' for S3. The unit difference between
`obj-type=='signals'` (per signal) and `obj-type=='shots'` (per file or per
store) noted above applies here too.

Big-EC2 per-signal / per-store read-time estimates, reusing the helpers
defined above. Unit handling matches the small-EC2 case.

In [ ]:
persig_big = pd.DataFrame(
    [
        (
            "HDF5",
            "local",
            "signals",
            "s / signal",
            _est_signals_mode(big_data, "local", "off"),
        ),
        (
            "HDF5",
            "local",
            "shots",
            "s / file",
            _est_shots_mode(big_data, "local", "off"),
        ),
        (
            "HDF5",
            "S3",
            "signals",
            "s / signal",
            _est_signals_mode(big_data, "S3", "70MB"),
        ),
        ("HDF5", "S3", "shots", "s / file", _est_shots_mode(big_data, "S3", "70MB")),
        (
            "Zarr",
            "local",
            "signals",
            "s / signal",
            _est_signals_mode(zarr_big_data, "local"),
        ),
        (
            "Zarr",
            "local",
            "shots",
            "s / store",
            _est_shots_mode(zarr_big_data, "local"),
        ),
        ("Zarr", "S3", "signals", "s / signal", _est_signals_mode(zarr_big_data, "S3")),
        ("Zarr", "S3", "shots", "s / store", _est_shots_mode(zarr_big_data, "S3")),
    ],
    columns=["format", "where", "obj-type", "unit", "est-read-time"],
)
persig_big


Side-by-side bar charts for the `m7i.48xlarge`, with the same layout used for
the `m5.4xlarge`: signals-mode estimates on a log Y axis, shots-mode estimates on a linear Y axis.


In [ ]:
psig_big = persig_big.query("`obj-type` == 'signals'")
psig_big.assign(
    scenario=(
        psig_big["where"] + " / " + psig_big["obj-type"] + " [" + psig_big["unit"] + "]"
    )
).hvplot.bar(
    x="scenario",
    y="est-read-time",
    by="format",
    rot=20,
).options(
    hooks=[_set_bar_log_bottom],
    show_grid=True,
    title="Estimated read time per signal -- signals mode (m7i.48xlarge)",
    ylabel="Estimated read time / [s/signal]  (log scale)",
    xlabel="Scenario (where / obj-type [unit])",
    height=400,
    width=560,
    logy=True,
    ylim=(1e-6, None),
)


In [ ]:
pst_big = persig_big.query("`obj-type` == 'shots'")
pst_big.assign(
    scenario=(
        pst_big["where"] + " / " + pst_big["obj-type"] + " [" + pst_big["unit"] + "]"
    )
).hvplot.bar(
    x="scenario",
    y="est-read-time",
    by="format",
    rot=20,
).options(
    show_grid=True,
    title="Estimated read time per file / per store -- shots mode (m7i.48xlarge)",
    ylabel="Estimated read time / [s/file or s/store]",
    xlabel="Scenario (where / obj-type [unit])",
    height=400,
    width=560,
    ylim=(0, None),
)
